In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import json

### Scrape Movie

In [8]:
# Scrapes the movie data from each individual url and returns a dictionary with the movie details
def scrape_movie(url):
    try:
        response = requests.get(url, timeout=10)
    except requests.exceptions.RequestException:
        return None

    soup = BeautifulSoup(response.text, 'html.parser')

    scripts = soup.find_all('script')

    movie_data = None

    for script in scripts:
        text = script.text

        if 'description' in text and 'genre' in text:
            try:
                movie_data = json.loads(text)
                break
            except json.JSONDecodeError:
                continue

    if movie_data is None:
        return None

    movie = {
        'title': movie_data.get('name'),
        'genres': movie_data.get('genre', []),
        'content_rating': movie_data.get('contentRating'),
        'release_date': movie_data.get('dateCreated'),
        'description': movie_data.get('description'),
        'audience_rating': movie_data.get('aggregateRating', {}).get('ratingValue'),
        'director': movie_data.get('director', [{}])[0].get('name'),
        'url': url
    }

    return movie

### Get movie urls

In [1]:
#Gets movie urls across genre by replacing the genre name into the api url to access the genre page then adding the end cursor to the url for pagination of the genre 
def get_movie_urls(genre,target=200):
    
    api_url = f"https://www.rottentomatoes.com/cnapi/browse/movies_at_home/genres:{genre}"
    
    base_url = "https://www.rottentomatoes.com"
    
    all_movie_urls = []
    cursor = None

    while len(all_movie_urls) < target:

        if cursor is None:
            url = api_url
        else:
            url = api_url + "?after=" + cursor

        response = requests.get(url)
        data = response.json()

        if 'grid' not in data:
            print('Unexpected response:', data)
            break

        for movie in data["grid"]["list"]:
            full_url = base_url + movie["mediaUrl"]

            if full_url not in all_movie_urls:
                all_movie_urls.append(full_url)

        if not data['pageInfo']['hasNextPage']:
            break

        cursor = data["pageInfo"]["endCursor"]

    return all_movie_urls

In [5]:
# A dictionary of genre according to its value in API
genres = {
    "Action": "action",
    "Adventure": "adventure",
    "Animation": "animation",
    "Comedy": "comedy",
    "Crime": "crime",
    "Documentary": "documentary",
    "Drama": "drama",
    "Fantasy": "fantasy",
    "Horror": "horror",
    "Mystery & Thriller": "mystery_and_thriller",
    "Romance": "romance",
    "Sci-Fi": "sci_fi",
    "War": "war",
    "Western": "western"
}

In [6]:
# get the multiple genres for each movie into a,list and save it in dictionary
all_genre_urls = {}

for genre_name, genre_api_name in genres.items():

    print(f"Scraping {genre_name}...")

    urls = get_movie_urls(genre_api_name)

    all_genre_urls[genre_name] = urls

    print(f"{genre_name}: {len(urls)} URLs")

Scraping Action...
Action: 210 URLs
Scraping Adventure...
Adventure: 210 URLs
Scraping Animation...
Animation: 210 URLs
Scraping Comedy...
Comedy: 210 URLs
Scraping Crime...
Crime: 210 URLs
Scraping Documentary...
Documentary: 210 URLs
Scraping Drama...
Drama: 210 URLs
Scraping Fantasy...
Fantasy: 210 URLs
Scraping Horror...
Horror: 210 URLs
Scraping Mystery & Thriller...
Mystery & Thriller: 210 URLs
Scraping Romance...
Romance: 210 URLs
Scraping Sci-Fi...
Unexpected response: {'error': 'NOT_FOUND', 'errorMessage': 'The request failed because it is not found', 'message': 'Not Found', 'status': 404}
Sci-Fi: 120 URLs
Scraping War...
War: 210 URLs
Scraping Western...
Western: 210 URLs


In [7]:
# Gets unique URLs and add its genre into its list
movie_genres = {}

for genre, urls in all_genre_urls.items():

    for url in urls:

        if url not in movie_genres:
            movie_genres[url] = []

        movie_genres[url].append(genre)

In [9]:
unique_movie_urls = list(movie_genres.keys())

print("Unique movie URLs:", len(unique_movie_urls))

Unique movie URLs: 1820


In [10]:
# Calles Scraping function and scrapes each and evry URL and get the movie data through it
all_movies = []

for url in unique_movie_urls:
    movie = scrape_movie(url)

    if movie is not None:
        all_movies.append(movie)

print("Movies scraped:", len(all_movies))

Movies scraped: 1820


In [11]:
df=pd.DataFrame(all_movies)

In [12]:
df.head()

,title,genres,content_rating,release_date,description,audience_rating,director,url
0,Batman: Knightfall - Part 1 - Knightfall,"[Drama, Crime, Mystery & Thriller, Adventure, ...",R,2026-08-24,"Discover reviews, ratings, and trailers for Ba...",97,Jeff Wamester,https://www.rottentomatoes.com/m/batman_knight...
1,Motor City,"[Action, Mystery & Thriller, Crime, Drama]",R,2026-07-24,"Discover reviews, ratings, and trailers for Mo...",63,Potsy Ponciroli,https://www.rottentomatoes.com/m/motor_city
2,Above and Below (2026),"[Mystery & Thriller, Action, Crime]",NaN,2026-07-29,"Discover reviews, ratings, and trailers for Ab...",50,Jesse V. Johnson,https://www.rottentomatoes.com/m/above_and_bel...
3,Facing El Chapo,"[Action, Crime, Drama]",NaN,2026-08-21,"Discover reviews, ratings, and trailers for Fa...",78,Chava Cartas,https://www.rottentomatoes.com/m/facing_el_chapo
4,Normal (2025),"[Action, Comedy, Crime, Mystery & Thriller]",R,2026-04-17,"Discover reviews, ratings, and trailers for No...",77,Ben Wheatley,https://www.rottentomatoes.com/m/normal_2025


In [14]:
df.to_csv(r"C:\Users\pc1\Downloads\rottentomatoes_movies.csv",index=False)